## Lorenz 63 path visualization

Loads trajectories from `datasets/LORENZ63/full_path/path_*.pt` (generated by `scripts/Lorenz63/generate_fields.py`).

Each file is a tensor shaped `(T, 1, 3, 1)` where:
- `T` = number of saved time frames (includes initial state at index 0)
- dimensions `[0,1,2]` = state variables `(x, y, z)`

Classic DA-benchmark parameters: `sigma=10`, `rho=28`, `beta=8/3`, `dt=0.01`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from project_utils.paths import dataset_trajectory_dir

# --- user settings (edit these) ---
DATA_ROOT = "LORENZ63"
TRAJECTORY_SUBDIR = "full_path"

PATH_IDS = [0]
VAR_LABELS = ["x", "y", "z"]

# Optional: restrict time window for plots (None = use all frames)
T_START = None
T_END = None

traj_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
paths = {}
for pid in PATH_IDS:
    fpath = os.path.join(traj_dir, f"path_{pid}.pt")
    traj = torch.load(fpath, map_location="cpu", weights_only=False)
    if traj.ndim == 4:
        traj = traj[:, 0, :, 0]  # (T, 3)
    elif traj.ndim == 2:
        pass
    else:
        raise ValueError(f"Unexpected shape {tuple(traj.shape)} in {fpath}")

    t0 = 0 if T_START is None else int(T_START)
    t1 = traj.shape[0] if T_END is None else int(T_END)
    paths[pid] = traj[t0:t1]
    print(
        f"path_{pid}: file={fpath}\n"
        f"  shape={tuple(paths[pid].shape)}  dtype={paths[pid].dtype}  "
        f"min/max={paths[pid].min().item():.4g}/{paths[pid].max().item():.4g}"
    )

STATE_DIM = next(iter(paths.values())).shape[1]
assert STATE_DIM == 3, f"Expected state dim 3, got {STATE_DIM}"
print(f"STATE_DIM={STATE_DIM}, variables: {VAR_LABELS}")

### Time series: x, y, z

In [ ]:
n_paths = len(paths)
fig, axs = plt.subplots(n_paths, 1, figsize=(10, 3 * n_paths), sharex=True, squeeze=False)

for row, (pid, traj) in enumerate(sorted(paths.items())):
    ax = axs[row, 0]
    t_axis = np.arange(traj.shape[0])
    for d, lab in enumerate(VAR_LABELS):
        ax.plot(t_axis, traj[:, d].numpy(), label=lab, linewidth=1.0)
    ax.set_ylabel("state")
    ax.set_title(f"path_{pid}: Lorenz 63")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", ncol=3, fontsize=8)

axs[-1, 0].set_xlabel("time index")
plt.tight_layout()
plt.show()

### Phase portraits (classic butterfly)

In [ ]:
pid = PATH_IDS[0]
traj = paths[pid].numpy()

fig = plt.figure(figsize=(12, 4))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax3.plot(traj[:, 0], traj[:, 1], traj[:, 2], linewidth=0.4, color="C0")
ax3.set_xlabel("x")
ax3.set_ylabel("y")
ax3.set_zlabel("z")
ax3.set_title(f"path_{pid}: 3D attractor")

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(traj[:, 0], traj[:, 2], linewidth=0.4, color="C1")
ax2.set_xlabel("x")
ax2.set_ylabel("z")
ax2.set_title(f"path_{pid}: x–z projection")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Compare paths: one variable across trajectories

In [ ]:
COMPARE_VAR = 0  # 0=x, 1=y, 2=z

fig, ax = plt.subplots(figsize=(10, 3.5))
for pid, traj in sorted(paths.items()):
    ax.plot(traj[:, COMPARE_VAR].numpy(), label=f"path_{pid}", linewidth=1.0, alpha=0.9)
ax.set_xlabel("time index")
ax.set_ylabel(VAR_LABELS[COMPARE_VAR])
ax.set_title(f"{VAR_LABELS[COMPARE_VAR]} across selected paths")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()